In [1]:
!pip install groq

In [2]:
from groq import Groq
from google.colab import userdata
import pandas as pd
import json

client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

MODEL = "openai/gpt-oss-120b"

print("NutriGuide AI connected!")

NutriGuide AI connected!


In [3]:
food_data = {
    "food": [
        "boiled egg",
        "milk",
        "banana",
        "rice",
        "chicken breast"
    ],
    "serving_g": [50, 100, 100, 100, 100],
    "calories": [78, 61, 89, 130, 165],
    "protein": [6.3, 3.2, 1.1, 2.7, 31],
    "carbs": [0.6, 4.8, 22.8, 28, 0],
    "fat": [5.3, 3.3, 0.3, 0.3, 3.6]
}

food_df = pd.DataFrame(food_data)

food_df

,food,serving_g,calories,protein,carbs,fat
0,boiled egg,50,78,6.3,0.6,5.3
1,milk,100,61,3.2,4.8,3.3
2,banana,100,89,1.1,22.8,0.3
3,rice,100,130,2.7,28.0,0.3
4,chicken breast,100,165,31.0,0.0,3.6


In [4]:
def get_food_nutrition(food_name):

    result = food_df[
        food_df["food"].str.lower() == food_name.lower()
    ]

    if result.empty:
        return {"error": f"{food_name} not found in database"}

    food = result.iloc[0]

    return {
        "food": food["food"],
        "serving_g": int(food["serving_g"]),
        "calories": float(food["calories"]),
        "protein": float(food["protein"]),
        "carbs": float(food["carbs"]),
        "fat": float(food["fat"])
    }

In [5]:
get_food_nutrition("banana")

{'food': 'banana',
 'serving_g': 100,
 'calories': 89.0,
 'protein': 1.1,
 'carbs': 22.8,
 'fat': 0.3}

In [6]:
conversation_history = [
    {
        "role": "system",
        "content": """
        You are NutriGuide AI, an intelligent Indian food nutrition assistant.

        Your responsibilities:
        1. Answer food nutrition questions.
        2. Retrieve nutrition values using the available database.
        3. Help users compare foods and understand nutrients.
        4. Maintain awareness of previous conversation messages.

        Context rules:
        - Use previous messages to understand the user.
        - Remember information shared earlier in this conversation.
        - Do not ask the user to repeat information already available.
        - Never invent nutrition values.
        - Be friendly and explain answers simply.
        """
    }
]

print("Conversation memory initialized!")

Conversation memory initialized!


In [10]:

def chat_with_agent(user_input):

    # 1. Add user message to conversation history
    conversation_history.append({
        "role": "user",
        "content": user_input
    })

    # 2. Send conversation history and tools to Groq
    response = client.chat.completions.create(
        model=MODEL,
        messages=conversation_history,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # 3. Check whether AI requested a tool
    if message.tool_calls:

        conversation_history.append(message)

        # 4. Execute the requested tool
        for tool_call in message.tool_calls:

            function_name = tool_call.function.name

            arguments = json.loads(
                tool_call.function.arguments
            )

            if function_name == "get_food_nutrition":

                result = get_food_nutrition(**arguments)

                print("TOOL EXECUTED:", result)

                conversation_history.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

        # 5. Ask AI to generate the final answer
        final_response = client.chat.completions.create(
            model=MODEL,
            messages=conversation_history,
            tools=tools
        )

        ai_response = final_response.choices[0].message.content

    else:
        ai_response = message.content

    # 6. Store AI response in conversation history
    conversation_history.append({
        "role": "assistant",
        "content": ai_response
    })

    return ai_response


In [23]:
print(chat_with_agent("Hi! My name is Hariharan. I am an IT graduate."))

Hi Harihan! 👋 Nice to meet you. How can I assist you with nutrition information today?


In [24]:
print(chat_with_agent("Hi! My name is Hariharan. I am an IT graduate."))

Hello again, Harihan! 👋 Hope you’re doing well. If you have any questions about the nutrition of Indian foods—or anything else—I’m here to help!


In [25]:
print(chat_with_agent(
    "Since you know my name, greet me and tell me how much protein is in 100g of banana."
))

TOOL EXECUTED: {'food': 'banana', 'serving_g': 100, 'calories': 89.0, 'protein': 1.1, 'carbs': 22.8, 'fat': 0.3}
Hi Harihan! 👋 Great to see you again.  

In a 100 gram portion of fresh banana, there’s **about 1.1 grams of protein**. Let me know if you’d like more details or have other foods you’re curious about!


In [15]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_food_nutrition",
            "description": "Retrieve nutrition values from the food database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "food_name": {
                        "type": "string",
                        "description": "Name of the food"
                    }
                },
                "required": ["food_name"]
            }
        }
    }
]

print("Nutrition tool registered!")

Nutrition tool registered!


In [16]:
print(chat_with_agent("Hi! My name is Aswini. I am an ECE graduate."))

TOOL EXECUTED: {'food': 'banana', 'serving_g': 100, 'calories': 89.0, 'protein': 1.1, 'carbs': 22.8, 'fat': 0.3}
Hi Aswini! 👋 Nice to meet you. 

In 100 g of fresh banana, there’s **about 1.1 g of protein**. If you need more details on other nutrients or have any other food‑related questions, just let me know!


In [30]:
print(chat_with_agent(
    "How much protein is in 100g of banana?"
))

TOOL EXECUTED: {'food': 'banana', 'serving_g': 100, 'calories': 89.0, 'protein': 1.1, 'carbs': 22.8, 'fat': 0.3}
In 100 g of fresh banana, the protein content is **about 1.1 grams**. Feel free to ask about any other nutrients or foods!


In [29]:
print("Last user message:")
print(conversation_history[-2])

print("\nLast assistant message:")
print(conversation_history[-1])

Last user message:
{'role': 'tool', 'tool_call_id': 'fc_b080732c-5d94-404f-a8d4-a673c9fda3f7', 'content': '{"food": "banana", "serving_g": 100, "calories": 89.0, "protein": 1.1, "carbs": 22.8, "fat": 0.3}'}

Last assistant message:
{'role': 'assistant', 'content': 'A 100‑gram serving of banana contains **approximately\u202f1.1\u202fgrams of protein**. If you’d like to explore other nutrients or compare with different foods, just let me know!'}


In [28]:
test_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You must use the get_food_nutrition tool to answer nutrition questions."
        },
        {
            "role": "user",
            "content": "How much protein is in 100g of banana?"
        }
    ],
    tools=tools,
    tool_choice="required"
)

print(test_response.choices[0].message)

ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to answer using the get_food_nutrition tool. Query for banana. Use tool.', tool_calls=[ChatCompletionMessageToolCall(id='fc_af9e6150-7c01-4190-9e60-52c4f601ca3f', function=Function(arguments='{"food_name":"banana"}', name='get_food_nutrition'), type='function')])


In [27]:
conversation_history[0]["content"] = """
You are NutriGuide AI, an intelligent Indian food nutrition assistant.

Rules:
1. For every question asking about food nutrition values,
   you MUST call the get_food_nutrition tool.
2. Never invent nutrition values from your own knowledge.
3. If a food is not available in the database, clearly say so.
4. Use previous conversation messages to understand the user.
5. Be friendly and explain answers simply.
"""

print("System instructions updated!")

System instructions updated!


In [31]:
print(chat_with_agent(
    "How much protein is in 100g of banana?"
))

TOOL EXECUTED: {'food': 'banana', 'serving_g': 100, 'calories': 89.0, 'protein': 1.1, 'carbs': 22.8, 'fat': 0.3}
A 100‑gram portion of banana provides roughly **1.1 grams of protein**. Let me know if you’d like any more nutrition info!
